# Consolidation Box Breakout on SPY
## Strategy Brief
This strategy aims to capitalize on price breakouts from a consolidation box, a period where the price trades within a narrow range. The signal is generated when the price breaks above or below this range, predicting a continuation in the breakout direction. Trades are entered on breakouts and exited when the price re-enters the box or hits a stop-loss. Historical testing shows potential for capturing significant trends.

## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the parameters for our strategy, including the length of the consolidation box and the breakout thresholds.

In [ ]:
# Configuration
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
BOX_LENGTH = 20  # Number of days to consider for the consolidation box
BREAKOUT_THRESHOLD = 0.01  # 1% breakout threshold
STOP_LOSS = 0.02  # 2% stop-loss from entry price

## PHASE 2 - Data Exploration
We will download historical data for SPY from Yahoo Finance, compute the consolidation box, and visualize it alongside the price data.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download('SPY', start=START_DATE, end=END_DATE)

# Compute consolidation box
rolling_max = data['Close'].rolling(window=BOX_LENGTH).max()
rolling_min = data['Close'].rolling(window=BOX_LENGTH).min()

# Plot price and consolidation box
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close')
plt.plot(rolling_max, label='Rolling Max', linestyle='--')
plt.plot(rolling_min, label='Rolling Min', linestyle='--')
plt.title('SPY Price with Consolidation Box')
plt.legend()
plt.show()

## PHASE 3 - Strategy Engineering
We define the breakout signal and the logic for entering and exiting trades based on the breakout and stop-loss conditions.

In [ ]:
# Generate breakout signals
breakout_up = data['Close'] > (1 + BREAKOUT_THRESHOLD) * rolling_max.shift(1)
breakout_down = data['Close'] < (1 - BREAKOUT_THRESHOLD) * rolling_min.shift(1)

# Entry and exit logic
positions = pd.Series(0, index=data.index)
positions[breakout_up] = 1
positions[breakout_down] = -1

# Apply stop-loss
for i in range(1, len(data)):
    if positions[i-1] == 1 and data['Close'][i] < data['Close'][i-1] * (1 - STOP_LOSS):
        positions[i] = 0
    elif positions[i-1] == -1 and data['Close'][i] > data['Close'][i-1] * (1 + STOP_LOSS):
        positions[i] = 0

## PHASE 4 - Coding & Backtesting
We backtest the strategy by calculating daily returns based on the positions and plot the resulting equity curve.

In [ ]:
# Shift positions to avoid lookahead bias
positions = positions.shift(1).fillna(0)

# Calculate daily returns
returns = data['Close'].pct_change().fillna(0)
strategy_returns = positions * returns

# Compute equity curve
equity_curve = (1 + strategy_returns).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(equity_curve, label='Equity Curve')
plt.title('Strategy Equity Curve')
plt.legend()
plt.show()

## PHASE 5 - Performance Evaluation
We evaluate the strategy's performance using key metrics such as CAGR, Sharpe Ratio, Sortino Ratio, Calmar Ratio, and maximum drawdown, and compare it to a buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(returns):
    cagr = (equity_curve[-1] ** (252 / len(returns))) - 1
    sharpe_ratio = np.mean(returns) / np.std(returns) * np.sqrt(252)
    downside_returns = returns[returns < 0]
    sortino_ratio = np.mean(returns) / np.std(downside_returns) * np.sqrt(252)
    max_drawdown = (equity_curve.cummax() - equity_curve).max()
    calmar_ratio = cagr / max_drawdown
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

strategy_metrics = calculate_performance_metrics(strategy_returns)
buy_and_hold_metrics = calculate_performance_metrics(returns)

# Print comparison table
metrics_df = pd.DataFrame({'Strategy': strategy_metrics, 'Buy and Hold': buy_and_hold_metrics},
                          index=['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'])
print(metrics_df)

## PHASE 6 - Deploy & Monitor
We create a function to download the last 60 days of SPY data, compute today's signal, and print the current position.

In [ ]:
def get_current_signal():
    recent_data = yf.download('SPY', period='60d')
    recent_rolling_max = recent_data['Close'].rolling(window=BOX_LENGTH).max()
    recent_rolling_min = recent_data['Close'].rolling(window=BOX_LENGTH).min()
    
    if recent_data['Close'][-1] > (1 + BREAKOUT_THRESHOLD) * recent_rolling_max[-2]:
        print('Current Position: Long')
    elif recent_data['Close'][-1] < (1 - BREAKOUT_THRESHOLD) * recent_rolling_min[-2]:
        print('Current Position: Short')
    else:
        print('Current Position: Neutral')

get_current_signal()